In [ ]:
import pandas as pd
import pvlib
import pandas as pd
import numpy as np
import os 
import pvlib
from pvlib.location import Location

from sirta_utils.irradiance_tools import get_elevation, get_sunrise_sunset

### Get clear-sky irradiance at CUEE station

In [ ]:
# Given coordinates
lat  = 48.71727475470337  #13.73677405
long = 2.2089178853229936 #100.5321224

Year_ = 2023  # parameter, default 2023 (release preprocessing/ ships this default)

file_path = "SIRTA_irradiance_solys2_ICPH-daytime" 
irradiance_file = os.path.join(file_path, "%04d-cleaned.csv" % (Year_))


#start_time =  "05:00"
#stop_time  =  "22:00"

start_date ='%04d-%02d-%02d' % (Year_, 1,   1) 
end_date   ='%04d-%02d-%02d' % (Year_+1, 1, 1) 

defined_alt = 159.0  # SIRTA site elevation (m), confirmed 2026-09-28 by querying
# api.opentopodata.org/v1/srtm30m for (lat, long) above (get_elevation()'s own
# call, same site) and by recomputing Iclr for several January 2023 minutes at
# this elevation: matches sirta_data_2023.csv's shipped Iclr to float round-off
# (~1e-17), vs. up to 0.06 W/m^2 off using the previously-commented 13.0 value.
# Hardcoded (not a live get_elevation() call) so a rebuild does not depend on
# opentopodata.org being reachable or returning the same value in the future.


tz_name = 'Europe/Paris'
sirta   = Location(lat, long, tz_name, defined_alt, 'SIRTA') 

times = pd.date_range(start=start_date, end=end_date, freq='1min', tz=sirta.tz) 

solpos = pvlib.solarposition.get_solarposition(times, lat, long)
apparent_zenith = solpos['apparent_zenith']

airmass  = pvlib.atmosphere.get_relative_airmass(apparent_zenith, model="gueymard2003")
pressure = pvlib.atmosphere.alt2pres(defined_alt)
airmass  = pvlib.atmosphere.get_absolute_airmass(airmass, pressure)
 
linke_turbidity = pvlib.clearsky.lookup_linke_turbidity(times, lat, long)
dni_extra = pvlib.irradiance.get_extra_radiation(times) 
cs        = pvlib.clearsky.ineichen(apparent_zenith, airmass, linke_turbidity, defined_alt, dni_extra) 
cs        = cs.tz_convert(tz_name).tz_localize(None)  

In [ ]:
import matplotlib.pyplot as plt
import datetime

month = 8
cs.plot() 
plt.xlim([datetime.date(Year_, month, 1), datetime.date(Year_,  month, 2)])

In [ ]:
solar_info_df = pd.DataFrame(columns=['Datetime', 'zna',  'Iclr'])
solar_info_df['Datetime'] = cs.index 
solar_info_df["Iclr"] = cs.ghi.values
solar_info_df["Idhi"] = cs.dhi.values
solar_info_df["zna"] = apparent_zenith.values 
solar_info_df["cos_zna"] = np.cos(apparent_zenith.values*np.pi/180)  


In [ ]:
import matplotlib.pyplot as plt
import datetime
month2str = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug", 9: "Sep", 10:"Oct", 11:"Nov", 12:"Dec"}

month = 12 
fig, ax = plt.subplots(figsize=(12, 4))   
ax.plot(solar_info_df['Datetime'].values, solar_info_df["Iclr"].values/200)
ax.plot(solar_info_df["Datetime"].values, solar_info_df["cos_zna"].values)
ax.set_xlabel('Datetime')
ax.set_ylabel('Irradiance')
ax.legend(loc='upper right')
ax.set_title('Clear-sky Irradiance in %s' % month2str[month])
ax.set_xlim([datetime.date(Year_, month, 1), datetime.date(Year_,  month, 10)])

plt.show()

In [ ]:
solar_info_df

In [ ]:
ghi_df = pd.read_csv(irradiance_file, parse_dates = ["Datetime"])

ghi_df["Datetime"] = pd.to_datetime(ghi_df["Datetime"]) 
ghi_df["Datetime"] = ghi_df["Datetime"].dt.tz_convert(tz_name)
ghi_df['Datetime'] = ghi_df['Datetime'].dt.tz_localize(None)

In [ ]:
ghi_df

In [ ]:
site_df = pd.merge(ghi_df, solar_info_df, on=['Datetime'], how='left') 

In [ ]:
month = 5
fig, ax = plt.subplots(figsize=(12, 4))   
ax.plot(site_df['Datetime'].values, site_df["Iclr"].values)
ax.plot(site_df["Datetime"].values, site_df["I"].values)
ax.plot(site_df["Datetime"].values, site_df["Diff"].values)
ax.set_xlabel('Datetime')
ax.set_ylabel('Irradiance')
ax.legend(loc='upper right')
ax.set_title('Clear-sky Irradiance in %s' % month2str[month])
ax.set_xlim([datetime.date(Year_, month, 1), datetime.date(Year_,  month, 3)])

plt.show()

### Compute K_bar

In [ ]:
site_df 

In [ ]:
site_df['k'] = site_df['I'] / (site_df['Iclr'] + 1e-6)   
site_df['Date'] = site_df['Datetime'].dt.date
site_df['Time'] = site_df['Datetime'].dt.time

In [ ]:
site_df = get_sunrise_sunset(site_df, lat, long, defined_alt, tz_name)
site_df.set_index('Datetime', inplace=True) 
site_df = site_df[(site_df.index >=  site_df["sunrise"]) & (site_df.index <=  site_df["sunset"])]

In [ ]:
site_df

In [ ]:
site_df = site_df.reset_index()

In [ ]:
site_df

In [ ]:
# Calculate zna from cos_zna
site_df['zna'] = site_df['zna']

# Filter the DataFrame for the desired time range and zna condition
mask =  (site_df['zna'] <= 85)
filtered_df = site_df[mask]

# Replace Inf and -Inf with NaN using .loc to avoid SettingWithCopyWarning
filtered_df.loc[:, 'k'] = filtered_df['k'].replace([np.inf, -np.inf], np.nan)

# Calculate mean of 'k' for each date ignoring NaN and Inf values
mean_k = filtered_df.groupby('Date')['k'].mean().reset_index()
mean_k.rename(columns={'k': 'k_bar'}, inplace=True)

# Merge the mean back into the original DataFrame
site_df = site_df.merge(mean_k, on='Date', how='left')

# Drop the helper columns
site_df.drop(columns=['Time', 'k', 'zna'], inplace=True)

In [ ]:
site_df = site_df.dropna() 
 

In [ ]:
site_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(site_df['sunrise'].dt.date.values.tolist(), site_df['sunrise'].dt.hour.values.tolist())
ax.plot(site_df['sunrise'].dt.date.values.tolist(), site_df['sunset'].dt.hour.values.tolist()) 
ax.set_xlabel('Datetime')
ax.set_ylabel('Time')
ax.legend(loc='upper right')
ax.set_xlim([datetime.date(Year_, 1, 1), datetime.date(Year_, 12, 31)])
plt.show()

Here we will generate a column Date so that we can use it to merge with `cluster_site_df` ... 

## Splitting sky conditions

In [ ]:
from solar_utils.skycondition_splitter import split_sky_condition
pd.options.mode.chained_assignment = None  
import importlib
importlib.reload(importlib.import_module('solar_utils.skycondition_splitter'))


method='k_bar'
min_threshold_k=0.6

clr_threshold_condition=0.8
partly_threshold_k=0.6


clr_date_list, partly_cloudy_date_list, cloudy_date_list, average_k_list = split_sky_condition(site_df.reset_index(), method=method, min_threshold_k=min_threshold_k, clr_threshold_condition=clr_threshold_condition, partly_threshold_k=partly_threshold_k)
cluster_site_df = pd.DataFrame({'Date': clr_date_list + partly_cloudy_date_list + cloudy_date_list, 'condition': ['clr']*len(clr_date_list) + ['partly_cloudy']*len(partly_cloudy_date_list) + ['cloudy']*len(cloudy_date_list) } )


In [ ]:
np.unique(cluster_site_df["condition"].values)

In [ ]:
cluster_site_df

In [ ]:
merged_result = pd.merge(site_df, cluster_site_df, on="Date", how="left")  
#merged_result.drop(columns=['Date'], inplace=True)

In [ ]:
merged_result[["Datetime", "I"]][:1000]

In [ ]:
file_path

In [ ]:
merged_result.keys()

In [ ]:
basename_ = os.path.basename(irradiance_file).split(".csv")[0]
merged_result_file_name = basename_ + "_sky-cond.csv" 

merged_result.to_csv(os.path.join(file_path, merged_result_file_name))
 

Prepare for plotting

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(merged_result['Date'].values.tolist(),merged_result['I'].values.tolist(),  label = "I") 
ax.plot(merged_result['Date'].values.tolist(),merged_result['Iclr'].values.tolist(),  label = "Iclr", alpha=0.5)  
ax.set_xlabel('Datetime')
ax.set_ylabel('Irradiance')
ax.legend(loc='upper right')
ax.set_xlim([datetime.date(Year_, 1, 1), datetime.date(Year_, 12, 31)])

In [ ]:
month = 10
date  = 2
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(merged_result['Datetime'].values,merged_result['I'].values,  label = "I") 
ax.plot(merged_result['Datetime'].values,merged_result['Iclr'].values,  label = "Iclr", alpha=0.5)  
ax.plot(merged_result['Datetime'].values,merged_result['Diff'].values,  label = "Diff") 
ax.plot(merged_result['Datetime'].values,merged_result['Idhi'].values,  label = "Idhi", alpha=0.5)  
ax.set_xlabel('Datetime')
ax.set_ylabel('Irradiance')
ax.legend(loc='upper right')
ax.set_xlim([datetime.datetime(Year_, month, date, 0,0,0), datetime.datetime(Year_,  month, date+1, 23,0,0)])

In [ ]:
merged_result[merged_result['Datetime'].dt.date == datetime.date(Year_,10,2)]

In [ ]:
217626-216918